In [ ]:
# export GEMINI_API_KEY=xxx

In [ ]:
# Install dependencies (run once).
import sys
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13"


In [ ]:
import os, time
from google import genai
from google.genai import types as gtypes

# Load the API key. In Colab use the Secrets panel; locally use an environment variable.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
# Free-tier limits for this model, confirmed Sept 2026: 15 RPM, 250k TPM, 1000 RPD.
# Agent loops are bursty, so the per-minute cap is what the retry wrapper below absorbs;
# the daily cap is the one that ends a session.
MODEL = "gemini-3.1-flash-lite"

In [ ]:
def generate_with_retry(*, contents, config=None, max_attempts=6):
    """client.models.generate_content with exponential backoff on 429.

    Free-tier Gemini caps requests/minute. A ReAct loop can fire many calls
    back-to-back and trip the limit; we sleep and retry instead of crashing.
    """
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)

print(f"Gemini client ready (model={MODEL}).")

In [ ]:
# Smoke test: one chat call, no tools.
resp = generate_with_retry(
    contents="Say hello in one short sentence.",
)
print(resp.text)